In [3]:
import cv2
import mediapipe as mp

# -----------------------------
# GitHub Source Link
# -----------------------------
github_link = "https://github.com/ajanthadevi2012/"

mp_hands = mp.solutions.hands
draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

cap = cv2.VideoCapture(0)

# Camera properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

if fps == 0:
    fps = 30

# Save output in current folder
output_path = "finger_counter_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Mirror camera
    frame = cv2.flip(frame, 1)

    # MediaPipe processing
    result = hands.process(
        cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    )

    total = 0

    if result.multi_hand_landmarks:

        for hand in result.multi_hand_landmarks:

            lm = hand.landmark
            count = 0

            # Index, middle, ring and pinky
            for tip, pip in [
                (8, 6),
                (12, 10),
                (16, 14),
                (20, 18)
            ]:

                if lm[tip].y < lm[pip].y:
                    count += 1

            # Thumb
            if lm[4].x < lm[3].x:
                count += 1

            total += count

            # Draw hand landmarks
            draw.draw_landmarks(
                frame,
                hand,
                mp_hands.HAND_CONNECTIONS
            )

    # Display finger count
    cv2.putText(
        frame,
        f"Raised fingers: {total}",
        (20, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 0),
        2
    )

    # --------------------------------
    # Display GitHub source - Top Right
    # --------------------------------
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.5
    thickness = 1

    (text_width, text_height), _ = cv2.getTextSize(
        github_link,
        font,
        scale,
        thickness
    )

    x = width - text_width - 10
    y = 25

    cv2.putText(
        frame,
        github_link,
        (x, y),
        font,
        scale,
        (255, 255, 255),
        thickness
    )

    # Save processed frame
    out.write(frame)

    # Display live output
    cv2.imshow("Finger Counter", frame)

    # Press Q to stop
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# Release resources
cap.release()
out.release()
hands.close()
cv2.destroyAllWindows()

print("Video saved as:", output_path)

Video saved as: finger_counter_output.mp4
